
# Thí nghiệm 02 — Huấn luyện dự báo tốc độ gió CMIP6

## 1. Thiết lập môi trường và dữ liệu thực nghiệm

### 1.1. Mục tiêu

Notebook thực hiện huấn luyện và đánh giá bốn kiến trúc dự báo chuỗi thời gian trên dữ liệu tốc độ gió gần bề mặt từ bộ dữ liệu khí hậu CMIP6/CORDEX tại khu vực Đức.

Các mô hình được so sánh gồm:

1. LSTM.
2. PatchTST kết hợp toán tử Koopman.
3. Autoformer kết hợp toán tử Koopman.
4. Informer kết hợp toán tử Koopman.

Thí nghiệm khảo sát ảnh hưởng của độ dài chuỗi đầu vào và chân trời dự báo đến sai số của từng mô hình.

### 1.2. Bộ dữ liệu

Dữ liệu được đọc từ tệp đã chuẩn bị:

`data/wind_nearest_5sites.pt`

Tệp được tạo trong `01_climate_data.ipynb` và chứa:

- `wind_raw`: chuỗi tốc độ gió tại 5 vị trí, kích thước (2928, 5).
- `x_in`, `x_out`: các tensor cửa sổ trượt được chuẩn bị với cấu hình minh họa.
- Thông tin vị trí không gian và tham số tiền xử lý.

Trong Grid Search, các cửa sổ huấn luyện được tạo từ 400 thời điểm đầu của `wind_raw` cho từng cấu hình riêng biệt.

Vì vậy, Grid Search không sử dụng cố định các tensor `x_in` và `x_out` đã được lưu.

### 1.3. Kiến trúc mô hình

Các lớp mô hình được định nghĩa tại:

`02_climate_model.ipynb`

Notebook huấn luyện sử dụng trực tiếp những định nghĩa này mà không triển khai lại kiến trúc.

Các biến thể PatchTST, Autoformer và Informer đều kết hợp toán tử Koopman có ràng buộc ổn định.

LSTM được sử dụng làm mô hình đối chứng và không có thành phần Koopman.

### 1.4. Môi trường thực nghiệm

Thực nghiệm gốc đã được thực hiện với GPU và các checkpoint được lưu sau từng lượt huấn luyện.

Notebook hiện được tổ chức theo đường dẫn tương đối với thư mục project để thuận tiện cho việc đọc mã nguồn và quản lý kết quả.

Kết quả Grid Search được lưu tại:

`results/climate_grid_search_e4000/`

Thư mục này chứa checkpoint của các mô hình và tệp `metrics.csv`.

### 1.5. Nguyên tắc bảo toàn

Quá trình tái cấu trúc notebook chỉ thực hiện:

- Chuẩn hóa Markdown và comment.
- Làm rõ cấu hình huấn luyện.
- Tách mã nguồn trực quan hóa sang notebook Figures.
- Loại bỏ các biến hoặc cell không cần thiết khi không ảnh hưởng đến thực nghiệm.

Không sinh lại dữ liệu, huấn luyện lại mô hình hoặc ghi đè các checkpoint đã tồn tại.


In [ ]:
from pathlib import Path

import torch
import torch.nn as nn
import numpy as np
import pandas as pd


# 1. Xác định thư mục gốc của project
project_root = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (
            p / "experiments" / "02_climate" / "02_climate_model.ipynb"
        ).is_file()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Không tìm thấy thư mục gốc của project."
    )


# 2. Xác định đường dẫn các tệp đầu vào và kết quả
experiment_dir = project_root / "experiments" / "02_climate"

model_notebook = experiment_dir / "02_climate_model.ipynb"
dataset_path = experiment_dir / "data" / "wind_nearest_5sites.pt"

results_dir = (
    experiment_dir / "results" / "climate_grid_search_e4000"
)
metrics_path = results_dir / "metrics.csv"


# 3. Lựa chọn thiết bị tính toán
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# 4. Nạp các định nghĩa mô hình
get_ipython().run_line_magic(
    "run", f'"{model_notebook.as_posix()}"'
)


# 5. Đọc bộ dữ liệu đã được chuẩn bị
dataset = torch.load(
    dataset_path,
    map_location="cpu",
    weights_only=True
)

x_in = dataset["x_in"]
x_out = dataset["x_out"]

patch_len = dataset["patch_len"]
horizon = dataset["horizon"]
input_dim = x_in.shape[-1]


# 6. Hiển thị thông tin đầu vào
print("=== THÔNG TIN THỰC NGHIỆM CLIMATE ===")
print("Project:", project_root)
print("Device:", device)
print("Dataset:", dataset_path)

print("X:", x_in.shape)
print("Y:", x_out.shape)
print("Input dimension:", input_dim)
print("Patch length:", patch_len)
print("Forecast horizon:", horizon)

print("\nResults directory:", results_dir)
print("Results directory exists:", results_dir.is_dir())
print("Metrics file exists:", metrics_path.is_file())

if metrics_path.is_file():
    metrics = pd.read_csv(metrics_path)
    print("Completed records:", len(metrics))

=== CLIMATE EXPERIMENT ===
Device: cuda
X: torch.Size([310, 80, 5])
Y: torch.Size([310, 50])
Input dimension: 5
Patch length: 80
Forecast horizon: 10

=== CLIMATE EXPERIMENT - LOCAL ===
Project: c:\Users\Win 11 New\Desktop\pa gk
Device: cuda
Dataset: c:\Users\Win 11 New\Desktop\pa gk\experiments\02_climate\data\wind_nearest_5sites.pt
X: torch.Size([310, 80, 5])
Y: torch.Size([310, 50])
Grid Search results: c:\Users\Win 11 New\Desktop\pa gk\experiments\02_climate\results\climate_grid_search_e4000
Results directory exists: True
Metrics file exists: True
Completed training runs: 120


c:\Users\Win 11 New\Desktop\pa gk\.venv\lib\site-packages\torch\_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()



## 2. Hàm huấn luyện mô hình

### 2.1. Mục tiêu

Hàm `train()` thực hiện quá trình tối ưu trọng số cho một mô hình với một cấu hình độ dài đầu vào và chân trời dự báo cụ thể.

Hàm được sử dụng cho cả bốn mô hình trong Grid Search.

Quá trình huấn luyện sử dụng toàn bộ các cửa sổ dữ liệu của cấu hình hiện tại trong mỗi epoch (full-batch).

### 2.2. Hàm mất mát dự báo

Sai số dự báo được tính bằng Mean Squared Error (MSE):

$$
\mathcal{L}_{\mathrm{MSE}}
=
\frac{1}{N}
\sum_{i=1}^{N}
(\hat y_i-y_i)^2
$$

Trong đó:

- $\hat y_i$: giá trị dự báo.
- $y_i$: giá trị thực tế.
- $N$: tổng số phần tử trong tensor dự báo.

MSE đo sai số bình phương trung bình giữa đầu ra mô hình và nhãn dự báo.

### 2.3. Chính quy hóa Lyapunov

Đối với các biến thể có toán tử Koopman, hàm mất mát bổ sung thành phần chính quy hóa nhằm hạn chế sự gia tăng năng lượng trong không gian tiềm ẩn.

Gọi $z$ và $z'$ lần lượt là biểu diễn tiềm ẩn trước và sau toán tử Koopman.

Trong code, ma trận Lyapunov được chọn là:

$$
P=I
$$

Năng lượng được xét dưới dạng:

$$
V(z)=z^\top Pz
$$

Thành phần phạt:

$$
\mathcal{L}_{\mathrm{Lyap}}
=
\frac{1}{B}
\sum_{i=1}^{B}
\max\left(
0,
V(z'_i)-V(z_i)
\right)
$$

Trong đó $B$ là số mẫu của batch huấn luyện.

Phần phạt chỉ nhận giá trị dương khi năng lượng sau phép biến đổi lớn hơn năng lượng trước phép biến đổi.

### 2.4. Hàm mất mát tổng hợp

Hàm mục tiêu được sử dụng:

$$
\mathcal{L}
=
\mathcal{L}_{\mathrm{MSE}}
+
\lambda\mathcal{L}_{\mathrm{Lyap}}
$$

Với các biến thể Koopman:

$$
\lambda=0.1
$$

Đối với LSTM:

$$
\lambda=0
$$

Vì LSTM không có toán tử Koopman, mô hình đối chứng chỉ được tối ưu dựa trên MSE.

### 2.5. Quy trình huấn luyện

Mỗi epoch thực hiện:

1. Chuyển mô hình sang chế độ `train()`.
2. Xóa gradient của lượt cập nhật trước.
3. Thực hiện truyền xuôi và thu nhận dự báo cùng các biểu diễn tiềm ẩn.
4. Tính MSE và thành phần chính quy hóa Lyapunov.
5. Lan truyền ngược gradient.
6. Cập nhật tham số bằng Adam.
7. Ghi lại loss và chỉ số theo dõi tham số Koopman.

Sau khi hoàn thành các epoch, mô hình chuyển sang chế độ `eval()` và tạo dự báo trên chính tập cửa sổ vừa sử dụng để huấn luyện.

### 2.6. Giá trị trả về

Hàm `train()` trả về:

| Giá trị | Ý nghĩa |
|---|---|
| `preds` | Dự báo sau huấn luyện |
| `losses` | Lịch sử hàm mất mát theo epoch |
| `eigs` | Lịch sử chỉ số được tính từ tham số `S_raw` |

**Lưu ý:** Trong code, `eigs` được tính từ giá trị lớn nhất của `sigmoid(S_raw)`, chưa nhân với giới hạn 0.99.

Do đó, giá trị này không nên được gọi trực tiếp là trị riêng lớn nhất hay giá trị kỳ dị thực tế của ma trận Koopman nếu chưa xét đầy đủ phép tham số hóa.

### 2.7. Giới hạn đánh giá

Hàm tạo dự báo sau huấn luyện trên chính dữ liệu được sử dụng để tối ưu trọng số.

Vì vậy, các metric tính từ đầu ra này là kết quả trên dữ liệu huấn luyện, không phải đánh giá tổng quát hóa trên tập kiểm tra độc lập.


In [ ]:


def train(model, x_in, x_out, epochs=4000, lr=3e-4, koop_attr="koop", lyap_weight=0.1):

    model.to(device)
    x_in, x_out = x_in.to(device), x_out.to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    mse = nn.MSELoss()
    losses, eigs = [], []

    # Khởi tạo ma trận Lyapunov P = I
    with torch.no_grad():
        _, z, z_next = model(x_in[:1], return_latents=True)

    P = torch.eye(z.shape[1], device=device)

    for ep in range(epochs):
        model.train()
        opt.zero_grad()

        pred, z, z_next = model(x_in, return_latents=True)
        pred_loss = mse(pred, x_out)

        if lyap_weight:
            zp = torch.einsum("bi,ij->bj", z, P)
            znp = torch.einsum("bi,ij->bj", z_next, P)

            lyap = torch.relu(
                (znp * z_next).sum(1) -
                (zp * z).sum(1)
            ).mean()
        else:
            lyap = torch.zeros(1, device=device)

        loss = pred_loss + lyap_weight * lyap
        loss.backward()
        opt.step()

        losses.append(loss.item())

        obj = getattr(model, koop_attr, None) if koop_attr else None
        eig_raw = (
            obj.S_raw if isinstance(obj, StrictStableKoopmanOperator)
            else None
        )

        eigs.append(
            0.0 if eig_raw is None
            else torch.sigmoid(eig_raw).abs().max().item()
        )

        if ep % max(1, epochs // 10) == 0 or ep == epochs - 1:
            print(
                f"Epoch {ep:>4}/{epochs} "
                f"MSE {pred_loss.item():.5f} "
                f"Lyap {(lyap_weight * lyap).item():.5f}"
            )

    model.eval()
    with torch.no_grad():
        preds = model(x_in).cpu().numpy()

    return preds, np.array(losses), np.array(eigs)



## 3. Cấu hình Grid Search

### 3.1. Mục tiêu

Grid Search được sử dụng để đánh giá ảnh hưởng của độ dài chuỗi đầu vào và chân trời dự báo đối với chất lượng dự báo tốc độ gió.

Thí nghiệm sử dụng dữ liệu của năm vị trí không gian đã được trích xuất trong `01_climate_data.ipynb`.

### 3.2. Không gian tìm kiếm

Các giá trị được khảo sát:

**Độ dài đầu vào:**

$$
P\in\{80,90,100,110,120,130\}
$$

**Chân trời dự báo:**

$$
H\in\{10,15,20,25,30\}
$$

Số tổ hợp tham số:

$$
6\times5=30
$$

Với bốn mô hình, tổng số lượt huấn luyện:

$$
30\times4=120
$$

Mỗi tổ hợp được huấn luyện trong 4000 epoch.

### 3.3. Các siêu tham số chung

| Tham số | Giá trị |
|---|---|
| Số kênh | 5 |
| Số epoch | 4000 |
| Learning rate | 0.0003 |
| Bộ tối ưu | Adam |
| `d_model` | 48 |
| Random seed | 7 |
| Hệ số chính quy hóa Lyapunov | 0.1 đối với Koopman |

LSTM sử dụng hai tầng với kích thước hidden state bằng 64.

Các biến thể Transformer sử dụng kiến trúc được định nghĩa trong `02_climate_model.ipynb`.

### 3.4. Chuẩn bị dữ liệu cho từng cấu hình

Với mỗi cặp $(P,H)$, notebook thực hiện:

1. Chọn 400 thời điểm đầu từ `wind_raw`.
2. Chuẩn hóa Min-Max riêng theo từng kênh.
3. Tạo các cửa sổ đầu vào độ dài $P$.
4. Tạo các nhãn gồm $H$ bước tương lai.

Số cửa sổ ở mỗi cấu hình:

$$
N=400-P-H
$$

Ví dụ, với $P=130$ và $H=20$:

$$
N=400-130-20=250
$$

Kích thước đầu vào:

$$
X\in\mathbb{R}^{250\times130\times5}
$$

Kích thước nhãn:

$$
Y\in\mathbb{R}^{250\times100}
$$

### 3.5. Ý nghĩa của patch length

Trong cách triển khai của Grid Search này, `P` vừa xác định độ dài cửa sổ đầu vào, vừa được truyền làm `patch_len` vào backbone.

Do đó, đối với PatchTST và Informer có sử dụng patch, cửa sổ đầu vào được chuyển thành một patch duy nhất trước khi đi vào Transformer.

Đây là đặc điểm của mã nguồn thực nghiệm hiện tại và cần được giữ nguyên khi mô tả phương pháp.

### 3.6. Phạm vi đánh giá

Các kết quả MSE và MAE được tính bằng cách so sánh dự báo với nhãn của chính các cửa sổ huấn luyện.

Notebook không thực hiện chia Train/Test độc lập trong nhánh thực nghiệm này.

Kết quả Grid Search được dùng để phân tích mức độ khớp dữ liệu theo cấu hình, không phải kết quả kiểm tra khả năng tổng quát hóa.


In [ ]:
# 1. Không gian Grid Search
patch_lengths = [80, 90, 100, 110, 120, 130]
horizons = [10, 15, 20, 25, 30]

# 2. Siêu tham số huấn luyện
d_model = 48
epochs = 4000
learning_rate = 3e-4

# 3. Đọc dữ liệu tốc độ gió đã trích xuất
wind_raw = dataset["wind_raw"].cpu().numpy()

# 4. Xác định thư mục lưu checkpoint và metrics
# results_dir đã được thiết lập tại cell 2
results_dir.mkdir(parents=True, exist_ok=True)
metrics_path = results_dir / "metrics.csv"

# 5. Tổng hợp cấu hình
print("=== CLIMATE WIND SPEED GRID SEARCH ===")

print("Patch lengths:", patch_lengths)
print("Horizons:", horizons)
print("d_model:", d_model)
print("Epochs:", epochs)

print(
    "Total configurations:",
    len(patch_lengths) * len(horizons)
)

print(
    "Total training runs:",
    len(patch_lengths) * len(horizons) * 4
)

=== CLIMATE GRID SEARCH ===
Patch lengths: [80, 90, 100, 110, 120, 130]
Horizons: [10, 15, 20, 25, 30]
d_model: 48
Epochs: 4000
Total configurations: 30
Total training runs: 120



## 4. Huấn luyện toàn bộ cấu hình Grid Search

### 4.1. Quy trình thực nghiệm

Vòng lặp Grid Search thực hiện lần lượt tất cả tổ hợp:

$$
(P,H,M)
$$

Trong đó:

- $P$: độ dài chuỗi đầu vào.
- $H$: chân trời dự báo.
- $M$: một trong bốn mô hình.

Tổng số lượt huấn luyện là 120.

### 4.2. Khởi tạo mô hình

Đối với từng cấu hình, notebook khởi tạo mô hình mới với các tham số tương ứng.

Cấu hình cụ thể:

| Mô hình | Đặc điểm |
|---|---|
| LSTM | 2 tầng, hidden size 64 |
| PatchTST + Koopman | `d_model=48`, patch length bằng độ dài đầu vào |
| Autoformer + Koopman | `d_model=48`, sử dụng phân rã chuỗi |
| Informer + Koopman | `d_model=48`, patch length bằng độ dài đầu vào |

Các biến thể Koopman sử dụng chính quy hóa Lyapunov với hệ số 0.1.

LSTM không áp dụng chính quy hóa này.

### 4.3. Huấn luyện và đánh giá

Với mỗi mô hình:

1. Khởi tạo kiến trúc theo cấu hình hiện tại.
2. Huấn luyện 4000 epoch bằng hàm `train()`.
3. Tạo dự báo sau huấn luyện trong chế độ `eval()`.
4. Tính MSE và MAE trên tập cửa sổ đã sử dụng để huấn luyện.
5. Lưu checkpoint và các thông tin liên quan.

Chỉ số MAE được tính bằng:

$$
\mathrm{MAE}
=
\frac{1}{N}\sum_{i=1}^{N}
|\hat{y}_i-y_i|
$$

MSE được tính theo công thức đã trình bày trong mục 2.

### 4.4. Cơ chế sử dụng checkpoint

Trước mỗi lượt huấn luyện, notebook kiểm tra sự tồn tại của checkpoint tương ứng.

Nếu checkpoint đã tồn tại, code đọc MSE và MAE từ checkpoint thay vì thực hiện huấn luyện mới.

Nếu chưa tồn tại, code huấn luyện mô hình và lưu kết quả.

Cơ chế này hỗ trợ tiếp tục Grid Search sau khi bị gián đoạn.

Tuy nhiên, nó chỉ kiểm tra sự tồn tại của tệp và đọc các metric đã lưu, chưa thực hiện xác minh đầy đủ tính tương thích của dữ liệu và cấu hình trước khi bỏ qua một lượt huấn luyện.

### 4.5. Nội dung checkpoint

Mỗi checkpoint được đặt tên theo:

`{model}_p{patch_length}_h{horizon}.pt`

Ví dụ:

`patchtst_p130_h20.pt`

Checkpoint chứa:

| Trường | Nội dung |
|---|---|
| `model_state_dict` | Trọng số mô hình |
| `pred` | Kết quả dự báo |
| `truth` | Nhãn thực tế |
| `loss` | Lịch sử hàm mất mát |
| `eig` | Lịch sử chỉ số theo dõi tham số Koopman |
| `mse` | MSE sau huấn luyện |
| `mae` | MAE sau huấn luyện |
| `patch_len` | Độ dài đầu vào |
| `horizon` | Chân trời dự báo |
| `d_model` | Kích thước biểu diễn |
| `input_dim` | Số kênh |
| `epochs` | Số epoch |

### 4.6. Tệp tổng hợp kết quả

Các metric được ghi vào:

`results/climate_grid_search_e4000/metrics.csv`

Tệp bao gồm các cột:

| Cột | Ý nghĩa |
|---|---|
| `Model` | Tên mô hình |
| `PatchLen` | Độ dài đầu vào |
| `Horizon` | Chân trời dự báo |
| `d_model` | Kích thước embedding |
| `MSE` | Sai số bình phương trung bình |
| `MAE` | Sai số tuyệt đối trung bình |

Tệp CSV được cập nhật sau mỗi lượt hoàn thành để bảo toàn tiến độ Grid Search.

### 4.7. Kết quả thực nghiệm đã lưu

Output của lần thực nghiệm trước ghi nhận:

- 30 tổ hợp độ dài đầu vào và chân trời dự báo.
- 4 mô hình trong mỗi tổ hợp.
- Tổng cộng 120 lượt huấn luyện.
- 4000 epoch cho mỗi lượt.

Các checkpoint và tệp metrics hiện có được sử dụng trực tiếp cho phần tổng hợp kết quả và trực quan hóa.

Không cần thực thi lại Grid Search để lấy các số liệu này.


In [ ]:

import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from contextlib import redirect_stdout
from io import StringIO

# 1. Thiết lập hạt giống ngẫu nhiên
np.random.seed(7)
torch.manual_seed(7)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(7)

metrics_path = results_dir / "metrics.csv"
records = {}

# 2. Duyệt toàn bộ cấu hình Grid Search
for p in patch_lengths:
    for h in horizons:

        print(f"\n=== Patch Length: {p} | Horizon: {h} ===")

        # Chuẩn hóa và tạo cửa sổ trượt
        sub = MinMaxScaler().fit_transform(wind_raw[:400, :])

        X, Y = [], []
        for i in range(len(sub) - p - h):
            X.append(sub[i:i + p])
            Y.append(sub[i + p:i + p + h].T.reshape(-1))

        x_grid = torch.tensor(np.stack(X), dtype=torch.float32)
        y_grid = torch.tensor(np.stack(Y), dtype=torch.float32)

        F = x_grid.shape[-1]

        # 3. Khởi tạo mô hình theo cấu hình hiện tại
        model_factories = {
            "LSTM": lambda: SimpleLSTMForecaster(F, 64, 2, h),
            "PatchTST": lambda: Koopformer_PatchTST(
                F, p, h, patch_len=p, d_model=d_model
            ),
            "Autoformer": lambda: Koopformer_Autoformer(
                F, p, h, patch_len=p, d_model=d_model
            ),
            "Informer": lambda: Koopformer_Informer(
                F, p, h, patch_len=p, d_model=d_model
            )
        }

        for name, create_model in model_factories.items():
            checkpoint_path = results_dir / f"{name.lower()}_p{p}_h{h}.pt"
            key = (name, p, h)

            # Đọc kết quả từ checkpoint nếu đã tồn tại
            if checkpoint_path.exists():
                saved = torch.load(
                    checkpoint_path,
                    map_location="cpu",
                    weights_only=True
                )
                mse_value = float(saved["mse"])
                mae_value = float(saved["mae"])
                print(f"{name:<12} Loaded checkpoint")

            else:
                model = create_model()

                # 4. Huấn luyện bằng hàm train đã định nghĩa
                with redirect_stdout(StringIO()):
                    preds, losses, eigs = train(
                        model,
                        x_grid,
                        y_grid,
                        epochs=epochs,
                        lr=learning_rate,
                        koop_attr=None if name == "LSTM" else "koop",
                        lyap_weight=0.0 if name == "LSTM" else 0.1
                    )

                truth = y_grid.numpy()
                mse_value = float(np.mean((preds - truth) ** 2))
                mae_value = float(np.mean(np.abs(preds - truth)))

                # 5. Lưu trọng số, dự báo và lịch sử huấn luyện
                torch.save({
                    "model_state_dict": model.cpu().state_dict(),
                    "pred": torch.from_numpy(preds),
                    "truth": y_grid.cpu(),
                    "loss": torch.from_numpy(losses),
                    "eig": torch.from_numpy(eigs),
                    "mse": mse_value,
                    "mae": mae_value,
                    "patch_len": p,
                    "horizon": h,
                    "d_model": d_model,
                    "input_dim": F,
                    "epochs": epochs
                }, checkpoint_path)

                del model
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

                print(f"{name:<12} Training completed")

            print(f"  MSE: {mse_value:.6f} | MAE: {mae_value:.6f}")

            records[key] = {
                "Model": name,
                "PatchLen": p,
                "Horizon": h,
                "d_model": d_model,
                "MSE": mse_value,
                "MAE": mae_value
            }

            # Cập nhật bảng metrics sau mỗi lượt hoàn thành
            pd.DataFrame(records.values()).to_csv(
                metrics_path, index=False
            )

print("\n=== GRID SEARCH COMPLETED ===")
print("Completed runs:", len(records))
print("Metrics saved to:", metrics_path)



=== Patch Length: 80 | Horizon: 10 ===
LSTM         Training completed
  MSE: 0.006458 | MAE: 0.060318
PatchTST     Training completed
  MSE: 0.004726 | MAE: 0.051087
Autoformer   Training completed
  MSE: 0.007338 | MAE: 0.065195
Informer     Training completed
  MSE: 0.002175 | MAE: 0.034726

=== Patch Length: 80 | Horizon: 15 ===
LSTM         Training completed
  MSE: 0.009620 | MAE: 0.074312
PatchTST     Training completed
  MSE: 0.003970 | MAE: 0.045991
Autoformer   Training completed
  MSE: 0.007571 | MAE: 0.066193
Informer     Training completed
  MSE: 0.003458 | MAE: 0.043394

=== Patch Length: 80 | Horizon: 20 ===
LSTM         Training completed
  MSE: 0.006216 | MAE: 0.059218
PatchTST     Training completed
  MSE: 0.005264 | MAE: 0.054053
Autoformer   Training completed
  MSE: 0.006985 | MAE: 0.063441
Informer     Training completed
  MSE: 0.003867 | MAE: 0.045930

=== Patch Length: 80 | Horizon: 25 ===
LSTM         Training completed
  MSE: 0.007728 | MAE: 0.066866
PatchTST